# Exercise 9 — Ordination Analysis
**AVCAD — Analysis and Visualisation of Complex Agro-Environmental Data**  
*Master in Data Science — Instituto Superior de Agronomia, Universidade de Lisboa*

---

### Objectives
Using the EFIplus_medit dataset, restricted to sites from the **Douro, Tejo, Mondego and Minho** basins:

1. Run a **PCA** based on quantitative environmental variables and produce a biplot with `Catchment_name` as grouping variable.
2. Run a **PCoA** and project the sites using the first two axes, with `Catchment_name` as grouping variable.
3. Run a **Linear Discriminant Analysis (LDA)** and produce a biplot with `Catchment_name` as grouping variable.


## 0. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import MDS
from sklearn.metrics import accuracy_score
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

# Colour palette for the four basins
palette = {
    'Douro':   '#E24B4A',
    'Tejo':    '#3B8BD4',
    'Mondego': '#1D9E75',
    'Minho':   '#F4A523'
}
basins = ['Douro', 'Tejo', 'Mondego', 'Minho']


## 1. Data Loading and Pre-processing

We load the EFIplus Mediterranean dataset and apply the same transformations identified in the previous exercise:
- **log1p** for `Altitude` and `Actual_river_slope` (right-skewed distributions)
- **sqrt** for `Elevation_mean_catch` (moderate skew)

Variables are then **standardized** (mean = 0, SD = 1) before ordination, as required when variables have different units.


In [ ]:
# Load dataset
df = pd.read_csv('EFIplus_medit.zip', compression='zip', sep=';')

# Filter to the four basins of interest
env_vars = ['Altitude', 'Actual_river_slope', 'Elevation_mean_catch',
            'prec_ann_catch', 'temp_ann', 'temp_jan', 'temp_jul']

sub = df[df['Catchment_name'].isin(basins)][['Catchment_name'] + env_vars].dropna().copy()

# Apply transformations (from Exercise 7/8)
sub['log1p_Altitude']            = np.log1p(sub['Altitude'])
sub['log1p_Actual_river_slope']  = np.log1p(sub['Actual_river_slope'])
sub['sqrt_Elevation_mean_catch'] = np.sqrt(sub['Elevation_mean_catch'])

trans_vars = [
    'log1p_Altitude', 'log1p_Actual_river_slope', 'sqrt_Elevation_mean_catch',
    'prec_ann_catch', 'temp_ann', 'temp_jan', 'temp_jul'
]

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(sub[trans_vars].values)
y = sub['Catchment_name'].values

print(f"Dataset: {X_scaled.shape[0]} sites x {X_scaled.shape[1]} variables")
print("\nSites per basin:")
print(sub['Catchment_name'].value_counts())


## 2. Principal Component Analysis (PCA)

PCA answers **Question #1**: how to reduce the *p* variables into a smaller number of latent variables 
that summarize most of the variability?

Each PC is a linear combination of the original variables, ordered by the amount of variance explained.  
The **biplot** simultaneously shows:
- **Object scores** — position of each site in the reduced space
- **Variable loadings** — arrows whose direction and length indicate the correlation of each variable with the PCs


In [ ]:
# Fit PCA
pca = PCA()
scores_pca = pca.fit_transform(X_scaled)
loadings    = pca.components_.T * np.sqrt(pca.explained_variance_)
var_exp     = pca.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Principal Component Analysis (PCA)', fontsize=14, fontweight='bold')

# --- Scree plot ---
ax = axes[0]
ax.bar(range(1, len(var_exp)+1), var_exp, color='steelblue', alpha=0.7, edgecolor='white')
ax.plot(range(1, len(var_exp)+1), np.cumsum(var_exp), 'ro-', lw=2, ms=5, label='Cumulative %')
ax.axhline(80, color='gray', ls='--', lw=1, label='80% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('Scree Plot')
ax.set_xticks(range(1, len(var_exp)+1))
ax.legend(fontsize=9)
for s in ['top','right']: ax.spines[s].set_visible(False)

# --- Biplot ---
ax = axes[1]
for basin in basins:
    mask = y == basin
    ax.scatter(scores_pca[mask, 0], scores_pca[mask, 1],
               c=palette[basin], alpha=0.5, s=25, label=basin)

short_names = ['Altitude', 'River slope', 'Elevation', 'Precipitation', 'Temp annual', 'Temp Jan', 'Temp Jul']
scale = 3.5
for i, var in enumerate(short_names):
    ax.annotate('', xy=(loadings[i,0]*scale, loadings[i,1]*scale), xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
    ax.text(loadings[i,0]*scale*1.13, loadings[i,1]*scale*1.13,
            var, fontsize=8, ha='center', fontweight='bold')

ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel(f'PC1 ({var_exp[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({var_exp[1]:.1f}%)')
ax.set_title(f'PCA Biplot  |  PC1 + PC2 = {var_exp[0]+var_exp[1]:.1f}% variance')
ax.legend(title='Basin', fontsize=9)
for s in ['top','right']: ax.spines[s].set_visible(False)

plt.tight_layout()
plt.show()

print(f"Variance explained: PC1={var_exp[0]:.1f}%  |  PC2={var_exp[1]:.1f}%  |  Cumulative={var_exp[0]+var_exp[1]:.1f}%")


### Interpretation — PCA

- **PC1 (60.2%)** captures the main environmental gradient, primarily driven by **temperature variables** (positive direction)
vs. **altitude and elevation** (negative direction). This reflects an elevation–temperature trade-off typical of river basins.
- **PC2 (22.2%)** is mainly associated with **precipitation**, distinguishing wetter from drier basins.
- Together, PC1 + PC2 explain **82.4%** of the total variance — two components are sufficient to represent the data well.
- **Minho** sites tend to have higher precipitation and lower temperatures (left/upper region), while **Tejo** sites 
are associated with higher temperatures (right region). **Douro** and **Mondego** overlap considerably.


## 3. Principal Coordinates Analysis (PCoA)

PCoA answers **Question #3**: how to represent the similarity among *n* objects in a minimum number of dimensions?

Unlike PCA (which works on the variance-covariance matrix), PCoA starts from a **dissimilarity matrix** between objects — 
making it flexible to use any distance metric. Here we use **Euclidean distance**, which makes the result equivalent to PCA 
(a useful comparison). With other metrics (e.g. Bray-Curtis for species data) PCoA would differ substantially from PCA.


In [ ]:
# Build Euclidean distance matrix and run metric MDS (= PCoA)
dist_matrix = squareform(pdist(X_scaled, metric='euclidean'))
pcoa = MDS(n_components=2, dissimilarity='precomputed', random_state=42, n_init=4)
scores_pcoa = pcoa.fit_transform(dist_matrix)

fig, ax = plt.subplots(figsize=(9, 7))
for basin in basins:
    mask = y == basin
    ax.scatter(scores_pcoa[mask, 0], scores_pcoa[mask, 1],
               c=palette[basin], alpha=0.5, s=25, label=basin)

ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel('PCoA Axis 1')
ax.set_ylabel('PCoA Axis 2')
ax.set_title('Principal Coordinates Analysis (PCoA)\nEuclidean distances — Douro, Tejo, Mondego, Minho',
             fontweight='bold')
ax.legend(title='Basin', fontsize=9)
for s in ['top','right']: ax.spines[s].set_visible(False)
plt.tight_layout()
plt.show()


### Interpretation — PCoA

The PCoA ordination produces a pattern **very similar to PCA**, as expected when using Euclidean distances on standardized data — 
both methods are mathematically equivalent in this case.  
The separation between basins along Axis 1 mirrors the temperature/altitude gradient seen in the PCA biplot.  
The advantage of PCoA is its flexibility: by switching to a non-Euclidean dissimilarity measure 
(e.g. Bray-Curtis for community data), it can capture patterns that PCA cannot.


## 4. Linear Discriminant Analysis (LDA)

LDA answers **Question #4**: what are the latent variables that **best discriminate** predetermined groups?

Unlike PCA (which maximizes total variance), LDA maximizes the **ratio of between-group to within-group variance**.  
It can also be used as a **classification method** to assign new observations to groups.

The number of discriminant functions (DFs) is at most min(n_groups − 1, n_variables) = min(3, 7) = **3 DFs**.


In [ ]:
# Fit LDA
lda = LinearDiscriminantAnalysis()
scores_lda = lda.fit_transform(X_scaled, y)
scalings    = lda.scalings_   # variable contributions to each DF
exp_var_lda = lda.explained_variance_ratio_ * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Linear Discriminant Analysis (LDA)', fontsize=14, fontweight='bold')

# --- DF scores scatter ---
ax = axes[0]
for basin in basins:
    mask = y == basin
    ax.scatter(scores_lda[mask, 0], scores_lda[mask, 1],
               c=palette[basin], alpha=0.5, s=25, label=basin)
    cx, cy = scores_lda[mask, 0].mean(), scores_lda[mask, 1].mean()
    ax.scatter(cx, cy, c=palette[basin], s=130, marker='*',
               edgecolors='black', lw=0.8, zorder=5)

ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel(f'DF1 ({exp_var_lda[0]:.1f}%)')
ax.set_ylabel(f'DF2 ({exp_var_lda[1]:.1f}%)')
ax.set_title('LDA scores  (★ = group centroids)')
ax.legend(title='Basin', fontsize=9)
for s in ['top','right']: ax.spines[s].set_visible(False)

# --- Biplot with loadings ---
ax = axes[1]
for basin in basins:
    mask = y == basin
    ax.scatter(scores_lda[mask, 0], scores_lda[mask, 1],
               c=palette[basin], alpha=0.4, s=20, label=basin)

scale = 2.5
short_names = ['Altitude', 'River slope', 'Elevation', 'Precipitation', 'Temp annual', 'Temp Jan', 'Temp Jul']
for i, var in enumerate(short_names):
    ax.annotate('', xy=(scalings[i,0]*scale, scalings[i,1]*scale), xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5))
    ax.text(scalings[i,0]*scale*1.15, scalings[i,1]*scale*1.15,
            var, fontsize=8, ha='center', color='darkred', fontweight='bold')

ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.axvline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel(f'DF1 ({exp_var_lda[0]:.1f}%)')
ax.set_ylabel(f'DF2 ({exp_var_lda[1]:.1f}%)')
ax.set_title('LDA Biplot  (arrows = variable contributions to discrimination)')
ax.legend(title='Basin', fontsize=9)
for s in ['top','right']: ax.spines[s].set_visible(False)

plt.tight_layout()
plt.show()

# Classification accuracy (resubstitution)
y_pred = lda.predict(X_scaled)
acc = accuracy_score(y, y_pred)
print(f"Variance explained per DF:")
for i, v in enumerate(exp_var_lda): print(f"  DF{i+1}: {v:.1f}%")
print(f"\nResubstitution classification accuracy: {acc:.3f} ({acc*100:.1f}%)")


### Interpretation — LDA

- **DF1 explains 79.9%** of the between-group discrimination. Temperature variables are the main drivers, 
separating **Tejo** (warmer, lower altitude) from **Minho** (cooler, higher precipitation).
- **DF2 (11.6%)** provides additional separation, mainly driven by precipitation, helping distinguish **Mondego**.
- The **resubstitution accuracy of 76.3%** means that when the same data used to train the model is used to classify, 
76.3% of sites are assigned to the correct basin. This is an optimistic estimate — cross-validation would give a more 
realistic figure.
- Compared to PCA, the LDA biplot shows **clearer separation between groups** because it explicitly maximizes 
between-group distances.

### Comparison: PCA vs PCoA vs LDA

| Method | Goal | Groups used? | Input |
|--------|------|-------------|-------|
| PCA | Maximize total variance | No | Covariance/correlation matrix |
| PCoA | Preserve pairwise distances | No | Dissimilarity matrix |
| LDA | Maximize between-group separation | Yes (supervised) | Raw data + group labels |
